In [1]:

# CELL 1: SETUP
# AI Privacy Guardian

import re
import json
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output
from datetime import datetime, timezone

print("AI Privacy Guardian setup completed!")
print("Python, Pandas, Matplotlib and interactive widgets are ready.")


AI Privacy Guardian setup completed!
Python, Pandas, Matplotlib and interactive widgets are ready.


In [2]:

# CELL 2: PRIVACY DETECTION ENGINE

PATTERNS = {
    "Email Address": (
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    ),
    "Indian Phone Number": (
        r"(?<!\d)(?:\+91[\s-]?)?[6-9]\d{9}(?!\d)"
    ),
    "Indian PAN Pattern": (
        r"\b[A-Z]{5}\d{4}[A-Z]\b"
    ),
    "Password": (
        r"(?i)\b(?:password|passwd|pwd)\s*[:=]\s*\S+"
    ),
    "API Key or Access Token": (
        r"(?i)\b(?:api[_ -]?key|access[_ -]?token)"
        r"\s*[:=]\s*[A-Za-z0-9_\-./]{8,}"
    )
}

WEIGHTS = {
    "Email Address": 15,
    "Indian Phone Number": 20,
    "Indian PAN Pattern": 30,
    "Password": 40,
    "API Key or Access Token": 40
}


def scan_text(text):
    """Find supported sensitive-data patterns without exposing values."""
    findings = []

    for category, pattern in PATTERNS.items():
        for match in re.finditer(pattern, text):
            findings.append({
                "category": category,
                "start": match.start(),
                "end": match.end()
            })

    # Sort and remove overlapping matches.
    findings.sort(key=lambda item: (item["start"], -item["end"]))

    filtered = []
    last_end = -1

    for item in findings:
        if item["start"] >= last_end:
            filtered.append(item)
            last_end = item["end"]

    return filtered


def redact_text(text, findings=None):
    """Replace detected values with category placeholders."""
    if findings is None:
        findings = scan_text(text)

    result = text

    for item in reversed(findings):
        placeholder = (
            "[" + item["category"].upper() + " REDACTED]"
        )
        result = (
            result[:item["start"]]
            + placeholder
            + result[item["end"]:]
        )

    return result


def calculate_risk(findings):
    """Rule-based score, not a validated security measurement."""
    raw_score = sum(
        WEIGHTS[item["category"]] for item in findings
    )
    score = min(100, raw_score)

    if score == 0:
        level = "LOW"
    elif score < 40:
        level = "MODERATE"
    elif score < 70:
        level = "HIGH"
    else:
        level = "CRITICAL"

    return score, level


def analyze_text(text):
    findings = scan_text(text)
    score, level = calculate_risk(findings)

    counts = {}
    for item in findings:
        category = item["category"]
        counts[category] = counts.get(category, 0) + 1

    return {
        "findings": findings,
        "counts": counts,
        "score": score,
        "level": level,
        "redacted_text": redact_text(text, findings)
    }


print("Privacy detection engine loaded successfully!")


Privacy detection engine loaded successfully!


In [3]:

# CELL 3: DEMO TEST

demo_text = """
Hello, my email is kavya.demo@example.com.
My phone number is 9876543210.
Password: DemoPass123!
Please protect my personal information before sharing.
"""

demo_result = analyze_text(demo_text)

print("PRIVACY RISK SCORE:", demo_result["score"], "/100")
print("RISK LEVEL:", demo_result["level"])
print("DETECTED CATEGORIES:", demo_result["counts"])

print("\nPROTECTED TEXT:")
print(demo_result["redacted_text"])


PRIVACY RISK SCORE: 75 /100
RISK LEVEL: CRITICAL
DETECTED CATEGORIES: {'Email Address': 1, 'Indian Phone Number': 1, 'Password': 1}

PROTECTED TEXT:

Hello, my email is [EMAIL ADDRESS REDACTED].
My phone number is [INDIAN PHONE NUMBER REDACTED].
[PASSWORD REDACTED]
Please protect my personal information before sharing.



In [4]:

# CELL 4: INTERACTIVE DASHBOARD

text_box = widgets.Textarea(
    value="",
    placeholder="Paste fictional test text here...",
    description="Text:",
    layout=widgets.Layout(width="100%", height="170px"),
    style={"description_width": "initial"}
)

scan_button = widgets.Button(
    description="Scan for Privacy Risks",
    button_style="primary",
    icon="search"
)

clear_button = widgets.Button(
    description="Clear",
    icon="trash"
)

output = widgets.Output()

def on_scan_clicked(_):
    with output:
        clear_output(wait=True)

        text = text_box.value

        if not text.strip():
            print("Please enter some text to scan.")
            return

        result = analyze_text(text)

        print("=" * 45)
        print("       AI PRIVACY GUARDIAN")
        print("=" * 45)
        print(f"Privacy risk score: {result['score']}/100")
        print(f"Risk classification: {result['level']}")
        print(f"Potential findings: {len(result['findings'])}")

        print("\nCATEGORY SUMMARY")
        if result["counts"]:
            summary = pd.DataFrame([
                {"Category": category, "Count": count}
                for category, count in result["counts"].items()
            ])
            display(summary)
        else:
            print("No supported patterns detected.")

        print("\nPROTECTED TEXT")
        print(result["redacted_text"])

        print("\nRECOMMENDATION")
        if result["level"] in ("HIGH", "CRITICAL"):
            print(
                "Review the detected information and remove "
                "secrets before sharing."
            )
        elif result["level"] == "MODERATE":
            print(
                "Review the detected information before sharing."
            )
        else:
            print(
                "No supported patterns were detected. "
                "This does not guarantee the text is safe."
            )

        if result["findings"]:
            counts = result["counts"]

            plt.figure(figsize=(8, 4))
            plt.bar(list(counts.keys()), list(counts.values()))
            plt.title("Potential Sensitive Information Detected")
            plt.xlabel("Category")
            plt.ylabel("Number of detections")
            plt.xticks(rotation=25, ha="right")
            plt.tight_layout()
            plt.show()

scan_button.on_click(on_scan_clicked)

def on_clear_clicked(_):
    text_box.value = ""
    with output:
        clear_output()

clear_button.on_click(on_clear_clicked)

display(widgets.HTML(
    "<h2>🔐 AI Privacy Guardian</h2>"
    "<p>Scan text, review potential risks, and redact "
    "selected sensitive patterns.</p>"
))
display(text_box)
display(widgets.HBox([scan_button, clear_button]))
display(output)


HTML(value='<h2>🔐 AI Privacy Guardian</h2><p>Scan text, review potential risks, and redact selected sensitive …

Textarea(value='', description='Text:', layout=Layout(height='170px', width='100%'), placeholder='Paste fictio…

Output()

In [5]:

# CELL 5: AUTOMATED TESTS

def run_tests():
    tests = [
        (
            "Email detection",
            lambda: any(
                x["category"] == "Email Address"
                for x in scan_text("Contact demo@example.com")
            )
        ),
        (
            "Phone detection",
            lambda: any(
                x["category"] == "Indian Phone Number"
                for x in scan_text("Call 9876543210")
            )
        ),
        (
            "Password detection",
            lambda: any(
                x["category"] == "Password"
                for x in scan_text("Password: TestPass123!")
            )
        ),
        (
            "Email masking",
            lambda: "demo@example.com" not in redact_text(
                "Contact demo@example.com"
            )
        ),
        (
            "Safe text has zero score",
            lambda: calculate_risk(
                scan_text("Hello, have a great day!")
            )[0] == 0
        ),
        (
            "Empty text has no findings",
            lambda: len(scan_text("")) == 0
        )
    ]

    passed = 0

    for name, test in tests:
        try:
            success = bool(test())
        except Exception:
            success = False

        print(
            ("PASS" if success else "FAIL") + " - " + name
        )
        passed += int(success)

    print(f"\nResult: {passed}/{len(tests)} tests passed")

run_tests()


PASS - Email detection
PASS - Phone detection
PASS - Password detection
PASS - Email masking
PASS - Safe text has zero score
PASS - Empty text has no findings

Result: 6/6 tests passed


In [6]:

# CELL 6: EXPORT REPORT

from google.colab import files

def export_privacy_report():
    text = text_box.value

    if not text.strip():
        print("Enter fictional test text in the dashboard first.")
        return

    result = analyze_text(text)

    report = {
        "project": "AI Privacy Guardian",
        "report_generated_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "risk_score": result["score"],
        "risk_level": result["level"],
        "total_detections": len(result["findings"]),
        "category_counts": result["counts"],
        "original_text_saved": False,
        "disclaimer": (
            "Heuristic prototype; detection is incomplete "
            "and the risk score is not a validated metric."
        )
    }

    filename = "privacy_guardian_report.json"

    with open(filename, "w", encoding="utf-8") as file:
        json.dump(report, file, indent=2)

    files.download(filename)

export_button = widgets.Button(
    description="Export Privacy Report",
    button_style="success",
    icon="download"
)

export_button.on_click(lambda _: export_privacy_report())

display(export_button)


Button(button_style='success', description='Export Privacy Report', icon='download', style=ButtonStyle())

Enter fictional test text in the dashboard first.


In [7]:

# ============================================================
# AI PRIVACY GUARDIAN
# Intelligent Personal Data Protection System
# Single-cell Google Colab project
# ============================================================

import re
import json
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from IPython.display import display, Markdown
import ipywidgets as widgets

# 1. PROJECT CONFIGURATION

PATTERNS = {
    "Email Address": (
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    ),
    "Indian Phone Number": (
        r"(?<!\d)(?:\+91[\s-]?)?[6-9]\d{9}(?!\d)"
    ),
    "Indian PAN Pattern": r"\b[A-Z]{5}\d{4}[A-Z]\b",
    "Password": (
        r"(?i)\b(?:password|passwd|pwd)\s*[:=]\s*\S+"
    ),
    "API Key or Access Token": (
        r"(?i)\b(?:api[_ -]?key|access[_ -]?token)"
        r"\s*[:=]\s*[A-Za-z0-9_\-./]{8,}"
    )
}

WEIGHTS = {
    "Email Address": 15,
    "Indian Phone Number": 20,
    "Indian PAN Pattern": 30,
    "Password": 40,
    "API Key or Access Token": 40
}


# 2. DETECTION ENGINE

def scan_text(text):
    findings = []

    for category, pattern in PATTERNS.items():
        for match in re.finditer(pattern, text):
            findings.append({
                "category": category,
                "start": match.start(),
                "end": match.end()
            })

    findings.sort(key=lambda x: (x["start"], -x["end"]))

    filtered = []
    last_end = -1

    for item in findings:
        if item["start"] >= last_end:
            filtered.append(item)
            last_end = item["end"]

    return filtered


def redact_text(text, findings):
    result = text

    for item in reversed(findings):
        placeholder = (
            "[" + item["category"].upper() + " REDACTED]"
        )
        result = (
            result[:item["start"]]
            + placeholder
            + result[item["end"]:]
        )

    return result


def analyze_text(text):
    findings = scan_text(text)

    score = min(
        100,
        sum(WEIGHTS[x["category"]] for x in findings)
    )

    if score == 0:
        level = "LOW"
    elif score < 40:
        level = "MODERATE"
    elif score < 70:
        level = "HIGH"
    else:
        level = "CRITICAL"

    counts = {}

    for item in findings:
        category = item["category"]
        counts[category] = counts.get(category, 0) + 1

    return {
        "findings": findings,
        "counts": counts,
        "score": score,
        "level": level,
        "redacted": redact_text(text, findings)
    }


# 3. AUTOMATED TESTS

def run_tests():
    tests = [
        ("Email detection",
         lambda: len(scan_text("demo@example.com")) == 1),

        ("Phone detection",
         lambda: any(
             x["category"] == "Indian Phone Number"
             for x in scan_text("9876543210")
         )),

        ("Password detection",
         lambda: any(
             x["category"] == "Password"
             for x in scan_text("Password: DemoPass123!")
         )),

        ("Email masking",
         lambda: "demo@example.com" not in redact_text(
             "demo@example.com",
             scan_text("demo@example.com")
         )),

        ("Safe text has zero score",
         lambda: analyze_text("Hello world!")["score"] == 0),

        ("Empty text has no findings",
         lambda: len(scan_text("")) == 0)
    ]

    results = []

    for name, test in tests:
        try:
            passed = bool(test())
        except Exception:
            passed = False

        results.append({"Test": name,
                        "Result": "PASS" if passed else "FAIL"})

    return results


# 4. DASHBOARD OUTPUT

display(Markdown(
    "# 🔐 AI Privacy Guardian\n"
    "### Intelligent Personal Data Protection System\n"
    "Privacy scanning • Risk assessment • Text redaction"
))

display(Markdown(
    "**Demo note:** This prototype uses fictional test data. "
    "Do not enter real passwords, tokens, or identity details."
))

text_input = widgets.Textarea(
    value=(
        "Hello, my email is kavya.demo@example.com.\n"
        "My phone number is 9876543210.\n"
        "Password: DemoPass123!\n"
        "Please protect my information before sharing."
    ),
    description="Input:",
    layout=widgets.Layout(width="100%", height="140px"),
    style={"description_width": "initial"}
)

scan_button = widgets.Button(
    description="Scan Privacy",
    button_style="primary",
    icon="search"
)

clear_button = widgets.Button(
    description="Clear",
    icon="trash"
)

export_button = widgets.Button(
    description="Export JSON Report",
    button_style="success",
    icon="download"
)

output = widgets.Output()
latest_report = {"data": None}


def scan_action(_):
    with output:
        output.clear_output(wait=True)

        text = text_input.value

        if not text.strip():
            print("Please enter text to scan.")
            latest_report["data"] = None
            return

        result = analyze_text(text)
        latest_report["data"] = result

        display(Markdown("## 📊 Privacy Scan Results"))

        print("Privacy Risk Score:", result["score"], "/100")
        print("Risk Level:", result["level"])
        print("Potential Findings:", len(result["findings"]))

        if result["counts"]:
            display(pd.DataFrame([
                {"Category": k, "Count": v}
                for k, v in result["counts"].items()
            ]))
        else:
            print("No supported patterns detected.")

        display(Markdown("### 🛡️ Protected Text"))
        print(result["redacted"])

        display(Markdown("### 📈 Detection Chart"))

        if result["counts"]:
            plt.figure(figsize=(9, 4))
            plt.bar(
                list(result["counts"].keys()),
                list(result["counts"].values())
            )
            plt.title("Potential Sensitive Data Detected")
            plt.xlabel("Category")
            plt.ylabel("Number of detections")
            plt.xticks(rotation=25, ha="right")
            plt.tight_layout()
            plt.show()
        else:
            print("No detection categories to plot.")

        display(Markdown("### 🧪 Automated Test Results"))

        test_results = run_tests()
        display(pd.DataFrame(test_results))

        passed = sum(
            row["Result"] == "PASS" for row in test_results
        )

        print(f"Tests passed: {passed}/{len(test_results)}")

        if result["level"] in ("HIGH", "CRITICAL"):
            print(
                "\nWARNING: Review and redact sensitive data "
                "before sharing."
            )
        elif result["level"] == "MODERATE":
            print(
                "\nReview detected information before sharing."
            )
        else:
            print(
                "\nNo supported patterns detected. "
                "This does not guarantee privacy safety."
            )


def clear_action(_):
    text_input.value = ""
    latest_report["data"] = None

    with output:
        output.clear_output()


def export_action(_):
    with output:
        if latest_report["data"] is None:
            print("Click Scan Privacy before exporting.")
            return

        result = latest_report["data"]

        report = {
            "project": "AI Privacy Guardian",
            "generated_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "risk_score": result["score"],
            "risk_level": result["level"],
            "total_findings": len(result["findings"]),
            "category_counts": result["counts"],
            "original_text_included": False,
            "disclaimer": (
                "Pattern-based prototype. Results may be "
                "incomplete; risk score is heuristic."
            )
        }

        filename = "AI_Privacy_Guardian_Report.json"

        with open(filename, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2)

        print("Report saved:", filename)

        try:
            from google.colab import files
            files.download(filename)
        except ImportError:
            print("Report created in the current Python session.")


scan_button.on_click(scan_action)
clear_button.on_click(clear_action)
export_button.on_click(export_action)

display(text_input)
display(widgets.HBox([
    scan_button, clear_button, export_button
]))
display(output)

# Run the initial demonstration automatically.
scan_action(None)

print("\n✅ AI PRIVACY GUARDIAN INITIALIZED SUCCESSFULLY")


# 🔐 AI Privacy Guardian
### Intelligent Personal Data Protection System
Privacy scanning • Risk assessment • Text redaction

**Demo note:** This prototype uses fictional test data. Do not enter real passwords, tokens, or identity details.

Textarea(value='Hello, my email is kavya.demo@example.com.\nMy phone number is 9876543210.\nPassword: DemoPass…

Output()


✅ AI PRIVACY GUARDIAN INITIALIZED SUCCESSFULLY
